<a href="https://colab.research.google.com/github/Urpia-S/Olist_E-commerce_Analytic-SQL-Python/blob/main/notebooks/01_preparacao_base_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 - Preparacao da base Olist no Colab

Neste notebook eu preparo a base que sera usada nas analises: baixo ou envio os CSVs, carrego tudo no SQLite e deixo as tabelas principais prontas para consulta.

## Como usar

Os CSVs sao baixados da Release `data-v1` do GitHub. Depois disso, o notebook monta o banco SQLite usado nos demais notebooks.

In [10]:
from pathlib import Path
import sqlite3
import urllib.request
import zipfile

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)

PROJECT_DIR = Path.cwd()
DATA_ZIP_URL = "https://github.com/Urpia-S/Olist_E-commerce_Analytic-SQL-Python/releases/download/data-v1/olist_csv_raw.zip"
RAW_DIR = PROJECT_DIR / "data" / "raw"
OUTPUT_DIR = PROJECT_DIR / "outputs_colab"
DB_PATH = PROJECT_DIR / "olist_colab.sqlite"

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Diretorio do projeto:", PROJECT_DIR)
print("Diretorio dos CSVs:", RAW_DIR)
print("Banco SQLite que sera criado:", DB_PATH)
print("Saidas CSV:", OUTPUT_DIR)

Diretorio do projeto: /content
Diretorio dos CSVs: /content/data/raw
Banco SQLite que sera criado: /content/olist_colab.sqlite
Saidas CSV: /content/outputs_colab


## 1. Localizacao dos dados

Primeiro defino quais arquivos espero encontrar e onde procurar por eles.

In [11]:
EXPECTED_FILES = {
    "olist_customers_dataset.csv": "stg_customers",
    "olist_geolocation_dataset.csv": "stg_geolocation",
    "olist_orders_dataset.csv": "stg_orders",
    "olist_order_items_dataset.csv": "stg_order_items",
    "olist_order_payments_dataset.csv": "stg_order_payments",
    "olist_order_reviews_dataset.csv": "stg_order_reviews",
    "olist_products_dataset.csv": "stg_products",
    "olist_sellers_dataset.csv": "stg_sellers",
    "product_category_name_translation.csv": "stg_product_category_translation",
}


def baixar_csvs_da_release():
    zip_path = RAW_DIR / "olist_csv_raw.zip"
    urllib.request.urlretrieve(DATA_ZIP_URL, zip_path)

    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(RAW_DIR)

    print("CSVs extraidos em:", RAW_DIR)


baixar_csvs_da_release()

CSV_PATHS = {name: RAW_DIR / name for name in EXPECTED_FILES}

pd.DataFrame(
    [{"arquivo": name, "tabela_staging": EXPECTED_FILES[name], "caminho": str(path)} for name, path in CSV_PATHS.items()]
)

CSVs extraidos em: /content/data/raw


,arquivo,tabela_staging,caminho
0,olist_customers_dataset.csv,stg_customers,/content/data/raw/olist_customers_dataset.csv
1,olist_geolocation_dataset.csv,stg_geolocation,/content/data/raw/olist_geolocation_dataset.csv
2,olist_orders_dataset.csv,stg_orders,/content/data/raw/olist_orders_dataset.csv
3,olist_order_items_dataset.csv,stg_order_items,/content/data/raw/olist_order_items_dataset.csv
4,olist_order_payments_dataset.csv,stg_order_payments,/content/data/raw/olist_order_payments_dataset...
5,olist_order_reviews_dataset.csv,stg_order_reviews,/content/data/raw/olist_order_reviews_dataset.csv
6,olist_products_dataset.csv,stg_products,/content/data/raw/olist_products_dataset.csv
7,olist_sellers_dataset.csv,stg_sellers,/content/data/raw/olist_sellers_dataset.csv
8,product_category_name_translation.csv,stg_product_category_translation,/content/data/raw/product_category_name_transl...


## 2. Conexao e funcoes auxiliares

Aqui ficam a conexao SQLite e pequenas funcoes para executar SQL, consultar resultados, salvar CSVs e plotar graficos.

In [12]:
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")


def executar_sql(script):
    conn.executescript(script)
    conn.commit()


def consulta(sql):
    return pd.read_sql_query(sql, conn)


def salvar_consulta(sql, arquivo):
    df = consulta(sql)
    destino = OUTPUT_DIR / arquivo
    df.to_csv(destino, index=False)
    print(f"Arquivo salvo: {destino}")
    return df


def grafico_barras(df, x, y, titulo, rotacao=0, top=None):
    dados = df.head(top) if top else df
    ax = dados.plot(kind="bar", x=x, y=y, legend=False, figsize=(10, 4))
    ax.set_title(titulo)
    ax.set_xlabel("")
    ax.set_ylabel(y)
    plt.xticks(rotation=rotacao, ha="right" if rotacao else "center")
    plt.tight_layout()
    plt.show()

## 3. Carga dos CSVs para staging

Carrego os arquivos crus em tabelas `stg_*`. Esta camada guarda os dados como chegaram dos CSVs.

In [13]:
def carregar_csv_para_sqlite(csv_path, table_name, chunksize=100_000):
    conn.execute(f'DROP TABLE IF EXISTS "{table_name}"')
    total = 0
    first_chunk = True

    for chunk in pd.read_csv(csv_path, chunksize=chunksize):
        chunk.to_sql(
            table_name,
            conn,
            if_exists="replace" if first_chunk else "append",
            index=False,
        )
        total += len(chunk)
        first_chunk = False

    conn.commit()
    return total


linhas_carregadas = []
for filename, table_name in EXPECTED_FILES.items():
    total = carregar_csv_para_sqlite(CSV_PATHS[filename], table_name)
    linhas_carregadas.append({"arquivo": filename, "tabela": table_name, "linhas": total})

pd.DataFrame(linhas_carregadas)

,arquivo,tabela,linhas
0,olist_customers_dataset.csv,stg_customers,99441
1,olist_geolocation_dataset.csv,stg_geolocation,1000163
2,olist_orders_dataset.csv,stg_orders,99441
3,olist_order_items_dataset.csv,stg_order_items,112650
4,olist_order_payments_dataset.csv,stg_order_payments,103886
5,olist_order_reviews_dataset.csv,stg_order_reviews,99224
6,olist_products_dataset.csv,stg_products,32951
7,olist_sellers_dataset.csv,stg_sellers,3095
8,product_category_name_translation.csv,stg_product_category_translation,71


## 4. Checagens iniciais

Antes de modelar, confiro volume de linhas, duplicidade nas chaves principais e distribuicao dos status de pedido.

In [14]:
consulta("""
-- Conto as linhas carregadas em cada tabela staging.
SELECT 'customers' AS tabela, COUNT(*) AS linhas FROM stg_customers
UNION ALL SELECT 'geolocation', COUNT(*) FROM stg_geolocation
UNION ALL SELECT 'orders', COUNT(*) FROM stg_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM stg_order_items
UNION ALL SELECT 'order_payments', COUNT(*) FROM stg_order_payments
UNION ALL SELECT 'order_reviews', COUNT(*) FROM stg_order_reviews
UNION ALL SELECT 'products', COUNT(*) FROM stg_products
UNION ALL SELECT 'sellers', COUNT(*) FROM stg_sellers
UNION ALL SELECT 'product_category_translation', COUNT(*) FROM stg_product_category_translation
ORDER BY tabela;
""")

,tabela,linhas
0,customers,99441
1,geolocation,1000163
2,order_items,112650
3,order_payments,103886
4,order_reviews,99224
5,orders,99441
6,product_category_translation,71
7,products,32951
8,sellers,3095


In [15]:
consulta("""
-- Verifico se as chaves principais vieram duplicadas.
SELECT 'customers.customer_id' AS chave, COUNT(*) - COUNT(DISTINCT customer_id) AS duplicatas FROM stg_customers
UNION ALL SELECT 'orders.order_id', COUNT(*) - COUNT(DISTINCT order_id) FROM stg_orders
UNION ALL SELECT 'products.product_id', COUNT(*) - COUNT(DISTINCT product_id) FROM stg_products
UNION ALL SELECT 'sellers.seller_id', COUNT(*) - COUNT(DISTINCT seller_id) FROM stg_sellers;
""")

,chave,duplicatas
0,customers.customer_id,0
1,orders.order_id,0
2,products.product_id,0
3,sellers.seller_id,0


In [16]:
consulta("""
-- Vejo a distribuicao dos pedidos por status.
SELECT order_status, COUNT(*) AS pedidos
FROM stg_orders
GROUP BY order_status
ORDER BY pedidos DESC;
""")

,order_status,pedidos
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


## 5. Modelo relacional

Transformo a staging em tabelas `core_*`, com tipos mais adequados para as analises.

In [17]:
modelagem_sql = """
-- Recrio a camada core do zero para deixar a execucao reprodutivel.
DROP TABLE IF EXISTS core_order_reviews;
DROP TABLE IF EXISTS core_order_payments;
DROP TABLE IF EXISTS core_order_items;
DROP TABLE IF EXISTS core_orders;
DROP TABLE IF EXISTS core_products;
DROP TABLE IF EXISTS core_sellers;
DROP TABLE IF EXISTS core_customers;
DROP TABLE IF EXISTS core_geolocation_zip_city;
DROP TABLE IF EXISTS core_product_category_translation;

-- Clientes: identificador unico, cidade, estado e prefixo de CEP.
CREATE TABLE core_customers AS
SELECT
    customer_id,
    customer_unique_id,
    CAST(NULLIF(customer_zip_code_prefix, '') AS INTEGER) AS customer_zip_code_prefix,
    LOWER(NULLIF(customer_city, '')) AS customer_city,
    NULLIF(customer_state, '') AS customer_state
FROM stg_customers;

-- Vendedores: localizacao do seller para analises operacionais.
CREATE TABLE core_sellers AS
SELECT
    seller_id,
    CAST(NULLIF(seller_zip_code_prefix, '') AS INTEGER) AS seller_zip_code_prefix,
    LOWER(NULLIF(seller_city, '')) AS seller_city,
    NULLIF(seller_state, '') AS seller_state
FROM stg_sellers;

-- Traducao das categorias para nomes em ingles quando disponivel.
CREATE TABLE core_product_category_translation AS
SELECT
    product_category_name,
    product_category_name_english
FROM stg_product_category_translation;

-- Produtos: categoria e atributos fisicos usados em analises por categoria.
CREATE TABLE core_products AS
SELECT
    product_id,
    NULLIF(product_category_name, '') AS product_category_name,
    CAST(NULLIF(product_name_lenght, '') AS INTEGER) AS product_name_length,
    CAST(NULLIF(product_description_lenght, '') AS INTEGER) AS product_description_length,
    CAST(NULLIF(product_photos_qty, '') AS INTEGER) AS product_photos_qty,
    CAST(NULLIF(product_weight_g, '') AS REAL) AS product_weight_g,
    CAST(NULLIF(product_length_cm, '') AS REAL) AS product_length_cm,
    CAST(NULLIF(product_height_cm, '') AS REAL) AS product_height_cm,
    CAST(NULLIF(product_width_cm, '') AS REAL) AS product_width_cm
FROM stg_products;

-- Pedidos: datas do funil de compra e entrega.
CREATE TABLE core_orders AS
SELECT
    order_id,
    customer_id,
    order_status,
    NULLIF(order_purchase_timestamp, '') AS order_purchase_timestamp,
    NULLIF(order_approved_at, '') AS order_approved_at,
    NULLIF(order_delivered_carrier_date, '') AS order_delivered_carrier_date,
    NULLIF(order_delivered_customer_date, '') AS order_delivered_customer_date,
    NULLIF(order_estimated_delivery_date, '') AS order_estimated_delivery_date
FROM stg_orders;

-- Itens: granularidade de produto dentro do pedido, com preco e frete.
CREATE TABLE core_order_items AS
SELECT
    order_id,
    CAST(NULLIF(order_item_id, '') AS INTEGER) AS order_item_id,
    product_id,
    seller_id,
    NULLIF(shipping_limit_date, '') AS shipping_limit_date,
    CAST(NULLIF(price, '') AS REAL) AS price,
    CAST(NULLIF(freight_value, '') AS REAL) AS freight_value
FROM stg_order_items;

-- Pagamentos: valores e forma de pagamento por pedido.
CREATE TABLE core_order_payments AS
SELECT
    order_id,
    CAST(NULLIF(payment_sequential, '') AS INTEGER) AS payment_sequential,
    NULLIF(payment_type, '') AS payment_type,
    CAST(NULLIF(payment_installments, '') AS INTEGER) AS payment_installments,
    CAST(NULLIF(payment_value, '') AS REAL) AS payment_value
FROM stg_order_payments;

-- Reviews: notas e comentarios dos clientes.
CREATE TABLE core_order_reviews AS
SELECT
    ROW_NUMBER() OVER (ORDER BY order_id, review_id, review_creation_date) AS review_row_id,
    review_id,
    order_id,
    CAST(NULLIF(review_score, '') AS INTEGER) AS review_score,
    NULLIF(review_comment_title, '') AS review_comment_title,
    NULLIF(review_comment_message, '') AS review_comment_message,
    NULLIF(review_creation_date, '') AS review_creation_date,
    NULLIF(review_answer_timestamp, '') AS review_answer_timestamp
FROM stg_order_reviews;

-- Geolocalizacao: consolido latitude/longitude por prefixo de CEP, cidade e estado.
CREATE TABLE core_geolocation_zip_city AS
SELECT
    CAST(NULLIF(geolocation_zip_code_prefix, '') AS INTEGER) AS geolocation_zip_code_prefix,
    LOWER(NULLIF(geolocation_city, '')) AS geolocation_city,
    NULLIF(geolocation_state, '') AS geolocation_state,
    AVG(CAST(NULLIF(geolocation_lat, '') AS REAL)) AS avg_lat,
    AVG(CAST(NULLIF(geolocation_lng, '') AS REAL)) AS avg_lng,
    COUNT(*) AS source_rows
FROM stg_geolocation
GROUP BY
    CAST(NULLIF(geolocation_zip_code_prefix, '') AS INTEGER),
    LOWER(NULLIF(geolocation_city, '')),
    NULLIF(geolocation_state, '');

-- Indices para acelerar joins e filtros mais usados.
CREATE UNIQUE INDEX IF NOT EXISTS customers_pk ON core_customers (customer_id);
CREATE UNIQUE INDEX IF NOT EXISTS sellers_pk ON core_sellers (seller_id);
CREATE UNIQUE INDEX IF NOT EXISTS products_pk ON core_products (product_id);
CREATE UNIQUE INDEX IF NOT EXISTS orders_pk ON core_orders (order_id);
CREATE INDEX IF NOT EXISTS customers_unique_id_idx ON core_customers (customer_unique_id);
CREATE INDEX IF NOT EXISTS orders_customer_id_idx ON core_orders (customer_id);
CREATE INDEX IF NOT EXISTS orders_purchase_timestamp_idx ON core_orders (order_purchase_timestamp);
CREATE INDEX IF NOT EXISTS order_items_product_id_idx ON core_order_items (product_id);
CREATE INDEX IF NOT EXISTS order_items_seller_id_idx ON core_order_items (seller_id);
CREATE INDEX IF NOT EXISTS order_reviews_order_id_idx ON core_order_reviews (order_id);
"""

executar_sql(modelagem_sql)

consulta("""
SELECT 'core_customers' AS tabela, COUNT(*) AS linhas FROM core_customers
UNION ALL SELECT 'core_orders', COUNT(*) FROM core_orders
UNION ALL SELECT 'core_order_items', COUNT(*) FROM core_order_items
UNION ALL SELECT 'core_order_payments', COUNT(*) FROM core_order_payments
UNION ALL SELECT 'core_order_reviews', COUNT(*) FROM core_order_reviews
UNION ALL SELECT 'core_products', COUNT(*) FROM core_products
UNION ALL SELECT 'core_sellers', COUNT(*) FROM core_sellers
ORDER BY tabela;
""")

,tabela,linhas
0,core_customers,99441
1,core_order_items,112650
2,core_order_payments,103886
3,core_order_reviews,99224
4,core_orders,99441
5,core_products,32951
6,core_sellers,3095


## 6. Views analiticas

Crio views para centralizar joins e calculos que aparecem em varios notebooks.

In [18]:
views_sql = """
-- Recrio as views para manter a camada analitica sincronizada com a core.
DROP VIEW IF EXISTS vw_order_items_enriched;
DROP VIEW IF EXISTS vw_order_delivery;
DROP VIEW IF EXISTS vw_reviews_por_pedido;
DROP VIEW IF EXISTS vw_order_finance;

-- Financeiro por pedido: agrego itens e pagamentos antes do join para evitar duplicar valores.
CREATE VIEW vw_order_finance AS
WITH itens AS (
    SELECT
        order_id,
        COUNT(*) AS quantidade_itens,
        SUM(price) AS receita_itens,
        SUM(freight_value) AS valor_frete
    FROM core_order_items
    GROUP BY order_id
),
pagamentos AS (
    SELECT order_id, SUM(payment_value) AS valor_pagamento
    FROM core_order_payments
    GROUP BY order_id
)
SELECT
    o.order_id,
    COALESCE(i.quantidade_itens, 0) AS quantidade_itens,
    COALESCE(i.receita_itens, 0) AS receita_itens,
    COALESCE(i.valor_frete, 0) AS valor_frete,
    COALESCE(p.valor_pagamento, 0) AS valor_pagamento
FROM core_orders o
LEFT JOIN itens i ON i.order_id = o.order_id
LEFT JOIN pagamentos p ON p.order_id = o.order_id;

-- Reviews por pedido: consolido nota media e marco avaliacoes baixas.
CREATE VIEW vw_reviews_por_pedido AS
SELECT
    order_id,
    ROUND(AVG(review_score), 2) AS nota_media_review,
    COUNT(*) AS quantidade_reviews,
    SUM(CASE WHEN review_score IN (1, 2) THEN 1 ELSE 0 END) AS reviews_baixos,
    CASE
        WHEN AVG(review_score) < 3 THEN 'baixa'
        WHEN AVG(review_score) = 3 THEN 'neutra'
        ELSE 'alta'
    END AS classe_satisfacao
FROM core_order_reviews
GROUP BY order_id;

-- Entrega por pedido: calculo tempos de aprovacao, transporte, entrega e atraso.
CREATE VIEW vw_order_delivery AS
SELECT
    o.order_id,
    o.customer_id,
    c.customer_unique_id,
    c.customer_state,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    CASE
        WHEN o.order_delivered_customer_date IS NOT NULL
        THEN CAST(julianday(date(o.order_delivered_customer_date)) - julianday(date(o.order_purchase_timestamp)) AS INTEGER)
    END AS dias_entrega,
    CASE
        WHEN o.order_approved_at IS NOT NULL
        THEN CAST(julianday(date(o.order_approved_at)) - julianday(date(o.order_purchase_timestamp)) AS INTEGER)
    END AS dias_ate_aprovacao,
    CASE
        WHEN o.order_delivered_carrier_date IS NOT NULL AND o.order_approved_at IS NOT NULL
        THEN CAST(julianday(date(o.order_delivered_carrier_date)) - julianday(date(o.order_approved_at)) AS INTEGER)
    END AS dias_ate_transportadora,
    CASE
        WHEN o.order_delivered_customer_date IS NOT NULL AND o.order_delivered_carrier_date IS NOT NULL
        THEN CAST(julianday(date(o.order_delivered_customer_date)) - julianday(date(o.order_delivered_carrier_date)) AS INTEGER)
    END AS dias_transporte_cliente,
    CASE
        WHEN o.order_delivered_customer_date IS NOT NULL
        THEN MAX(CAST(julianday(date(o.order_delivered_customer_date)) - julianday(date(o.order_estimated_delivery_date)) AS INTEGER), 0)
    END AS dias_atraso,
    CASE
        WHEN o.order_delivered_customer_date IS NULL THEN NULL
        WHEN date(o.order_delivered_customer_date) > date(o.order_estimated_delivery_date) THEN 1
        ELSE 0
    END AS entrega_atrasada
FROM core_orders o
JOIN core_customers c ON c.customer_id = o.customer_id;

-- Itens enriquecidos: junto pedido, cliente, vendedor, produto e categoria traduzida.
CREATE VIEW vw_order_items_enriched AS
SELECT
    oi.order_id,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,
    p.product_category_name,
    COALESCE(t.product_category_name_english, p.product_category_name, 'unknown') AS categoria_ingles,
    oi.price,
    oi.freight_value,
    o.order_purchase_timestamp,
    c.customer_unique_id,
    c.customer_state,
    s.seller_state
FROM core_order_items oi
JOIN core_orders o ON o.order_id = oi.order_id
JOIN core_customers c ON c.customer_id = o.customer_id
JOIN core_sellers s ON s.seller_id = oi.seller_id
LEFT JOIN core_products p ON p.product_id = oi.product_id
LEFT JOIN core_product_category_translation t ON t.product_category_name = p.product_category_name;
"""

executar_sql(views_sql)

consulta("SELECT * FROM vw_order_delivery LIMIT 5;")

,order_id,customer_id,customer_unique_id,customer_state,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,dias_entrega,dias_ate_aprovacao,dias_ate_transportadora,dias_transporte_cliente,dias_atraso,entrega_atrasada
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,SP,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,8,0,2,6,0,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,BA,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,14,2,0,12,0,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,GO,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,9,0,0,9,0,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,RN,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,14,0,4,10,0,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,SP,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,3,0,1,2,0,0


## Proximo passo

Com `olist_colab.sqlite` criado, sigo para os notebooks analiticos.